# MTH 4224 / CSE 4224 - Homework 2

## Splines, RBF Networks, SVMs, and Classifier Comparison

Deadline: announced in Canvas

Points: 55

### Instructions

- Submit your notebook, your three $M = 16$ videos from Problem 2, and optionally a PDF of handwritten work.
- Show all work on math problems.
- Explain your results and comment your code.

## Problem 1 - Splines Are Linear Basis Function Models (15 pts)

A cubic spline with knots $\xi_1 < \cdots < \xi_K$ is cubic between knots and has continuous first and second derivatives at each knot. Splines are exactly LBF models, $\hat f(x) = \theta^T h(x)$, for a clever choice of $h$.

### (a) Spline bases (5 pts)

1. With $(u)_+ = \max(u, 0)$, take the $K + 4$ basis functions

   $$h_0(x) = 1,\quad h_1(x) = x,\quad h_2(x) = x^2,\quad h_3(x) = x^3,\quad h_{3+k}(x) = (x - \xi_k)_+^3,\ \ k = 1, \ldots, K.$$

   Show that every $\hat f(x) = \sum_{m=0}^{K+3}\theta_m h_m(x)$ is a cubic spline.

   *Hint:* Confirm $\hat f(x)$ has degree-3 polynomial pieces with continuous first and second derivatives where they join.

2. B-splines are defined on knots $t_0 \leq t_1 \leq \cdots$ via the Cox-de Boor recursion:

   $$B_{i,0}(x) = \begin{cases}1 & t_i \leq x < t_{i+1}\\ 0 & \text{otherwise}\end{cases}, \qquad B_{i,p}(x) = \frac{x - t_i}{t_{i+p} - t_i}B_{i,p-1}(x) + \frac{t_{i+p+1} - x}{t_{i+p+1} - t_{i+1}}B_{i+1,p-1}(x).$$

   With the integer knots $t_i = i$, compute $B_{0,1}(x)$ and $B_{0,2}(x)$ as piecewise polynomials, and plot both.

### (b) Fit a B-spline solid (10 pts)

`hw2_vase_scan.csv` (loaded by the provided code) is a noisy 3D scan of a vase. Model its radius as a tensor-product spline

$$\hat r(\vartheta, z) = \sum_{j=1}^{m_\vartheta}\sum_{k=1}^{m_z} c_{jk}\, a_j(\vartheta)\, b_k(z),$$

where the $a_j$ and $b_k$ are cubic B-splines in the angle $\vartheta$ and the height $z$. Each design-matrix row is $a(\vartheta_i) \otimes b(z_i)$ (`np.kron`), so this is still an LBF model.

1. Convert the points to $(\vartheta, z, r)$: $\vartheta = \operatorname{atan2}(x_2, x_1)$, $z = x_3$, $r = \sqrt{x_1^2 + x_2^2}$.

2. Build both bases with `sklearn.preprocessing.SplineTransformer` (cubic), making the $\vartheta$ basis periodic (`extrapolation='periodic'`, fit on $[-\pi, \pi]$). Fit by least squares.

3. Split into training and test sets. For at least five knot grids from $5 \times 4$ to $65 \times 30$, tabulate the number of parameters and the training and test RMSE. What happens at the finest grid, and why?

4. Plot the raw point cloud in 3D, your best fitted vase as a 3D solid, and a heat map of its residuals over $(\vartheta, z)$.

## Problem 2 - Watching an RBF Network Learn (15 pts)

Use the Gaussian RBF network, loss, and notation from Lecture 6, with inputs $x, \xi_j \in \mathbb{R}^2$.

### (a) The geometry of a kernel (5 pts)

*Hint:* The gradients of $L$ with respect to $\xi_j$ and $\lambda_j$ were derived in the class notes, and you may use them.

1. Find the peak height of $\theta_j K_{\lambda_j}(\xi_j, x)$ and the radius at which it falls to half its peak.
2. Compute its volume, $\int_{\mathbb{R}^2} \theta_j K_{\lambda_j}(\xi_j, x)\,dx$. How do the height, width, and volume change as $\lambda_j$ grows?
3. Using the update $\xi_j \leftarrow \xi_j - \alpha \nabla_{\xi_j} L$, explain why:
   - a center with $\theta_j > 0$ moves toward where the model underpredicts (and one with $\theta_j < 0$ toward where it overpredicts)
   - a center stops once it reaches a critical point
   - a narrow kernel far from the data barely moves
4. Which points make $\lambda_j$ grow, and which make it shrink? Use the sign of $\frac{\lVert \xi_j - x_i\rVert^2}{\lambda_j^2} - 1$.

### (b) Train and watch (10 pts)

The provided code samples $y = \sin(x_1)\sin(x_2) + \varepsilon$ on $[-2\pi, 2\pi]^2$ (8 peaks and 8 valleys) and defines a helper, `liftedCircle`.

1. Using the latest `GaussianRBFnetwork` class from the notes, set the centers uniformly on the domain, $\lambda_j = 1.2$, and $\theta_j \sim \mathcal{N}(0, 0.1^2)$, with $\alpha \approx 10^{-3}$. Train with $M = 8, 16,$ and $32$, saving snapshots (`snapshot = 40`), and report the train/test MSE.

2. Write a function that plots a snapshot in two panels:
   - 3D: the model surface with the training points, each center on the surface, and its radius-$\lambda_j$ circle (use `liftedCircle`), colored by the sign of $\theta_j$.
   - Top-down: the true function shaded, the model's contours, each center's circle, and its path.

   Show the final frames for the three values of $M$ side by side.

3. Train $M = 16$ from three random initializations and make an `.mp4` of each. Does each kernel find its own peak or valley? Explain the failure modes you see using part (a).

## Problem 3 - The Kernel Trick (10 pts)

The provided code samples points around a torus, a tube of radius $r$ wrapped around a circle of radius $R$, labeling points inside $+1$ and outside $-1$. No plane in $\mathbb{R}^3$ separates them. It splits the data 60/20/20 into training, validation, and test sets; choose hyperparameters on the validation set.

### (a) Seeing the lift (2 pts)

Let $\rho = \sqrt{x_1^2 + x_2^2}$. In the $(\rho, x_3)$ half-plane, the torus is the disk $(\rho - R)^2 + x_3^2 < r^2$. Under $\phi(x) = \left(\rho,\ x_3,\ \rho^2 + x_3^2\right)$, show that the torus is exactly the set of points below the plane $v = 2R\rho - (R^2 - r^2)$, where $v$ is the third coordinate.

### (b) Fit the SVMs (8 pts)

1. Fit an RBF SVM and, for comparison, a degree-2 SVM (`kernel='poly', degree=2, gamma=1, coef0=1`), choosing `C` (and `gamma` for RBF) on the validation set. Report training, validation, and test accuracy.

2. Plot:
   - the data in $(\rho, x_3)$ coordinates;
   - the RBF boundary $h(x) = 0$ in 3D;
   - for both models, heat maps of $h(x)$ on at least three slices $x_3 = c$ and on $x_2 = 0$, with $h(x) = 0$ drawn.

3. What do the slices show that the 3D view hides? How does the degree-2 model fail?

## Problem 4 - Comparing Classifiers on Spam (15 pts)

Download the [UCI Spambase dataset](https://archive.ics.uci.edu/dataset/94/spambase): 4,601 emails with 57 features, each labeled spam or not spam. Split it 60/20/20 (`random_state = 2026`), stratified by class.

Compare Gaussian naive Bayes, logistic regression, LDA, QDA, $k$-nearest neighbors, a linear SVM, and an RBF SVM. Make every modeling choice with cross-validation on the training data, not with the test set.

1. (2 pts) Which is worse: a false positive (a real email marked spam) or a false negative (spam in the inbox)? Explain how this should affect how you evaluate the models.

2. (5 pts) Evaluate all seven classifiers at default settings with cross-validation, and plot the results.

3. (5 pts, with step 4) Tune the best two with cross-validation.

4. Evaluate the two tuned models once on the test set, with confusion matrices. Which would you deploy? Why?

5. (3 pts) Discuss your results:
   - Is the naive Bayes independence assumption reasonable?
   - Why does QDA struggle?
   - How did the false-positive versus false-negative question shape your choices?

## Provided code

In [ ]:
# Problem 1: load the vase scan
import numpy as np
import pandas as pd

# noisy 3D scan of a vase surface: columns x1, x2, x3
vase = pd.read_csv('../data/hw2_vase_scan.csv')
points = vase.to_numpy()
print(points.shape)

In [ ]:
# Problem 2: sample the RBF data
import numpy as np
from sklearn.model_selection import train_test_split

rng = np.random.default_rng(2026)
n = 1000

# sample inputs uniformly on the square [-2pi, 2pi] x [-2pi, 2pi]
X = rng.uniform(-2 * np.pi, 2 * np.pi, size = (n, 2))

# noisy outputs of z = sin(x1) sin(x2)
y = np.sin(X[:, 0]) * np.sin(X[:, 1]) + 0.1 * rng.standard_normal(n)

rbfTrainX, rbfTestX, rbfTrainY, rbfTestY = train_test_split(X, y, test_size = 0.25, random_state = 2026)

In [ ]:
# Problem 2(b) helper: a circle of radius lam around a center, lifted onto the model surface
def liftedCircle(model, center, lam, numPoints = 80, offset = 0.03):
    # angles around the circle
    t = np.linspace(0, 2 * np.pi, numPoints)

    # points on the circle in the (x1, x2) plane
    circle = center + lam * np.column_stack((np.cos(t), np.sin(t)))

    # heights on the current model surface, raised slightly so the curve stays visible
    height = model.predict(circle) + offset

    return circle[:, 0], circle[:, 1], height

# example: draw the circle for kernel j on a 3D axis ax
# ax.plot(*liftedCircle(model, model.xi[j], model.lam[j]), color = 'tab:red')

In [ ]:
# Problem 3: sample the torus data
import numpy as np
from sklearn.model_selection import train_test_split

R, r = 1.0, 0.45
rng = np.random.default_rng(2026)
n = 3000

# sample points uniformly in a box that contains the torus
X = rng.uniform([-1.7, -1.7, -0.8], [1.7, 1.7, 0.8], size = (n, 3))

# label points inside the torus +1 and outside -1
rho = np.sqrt(X[:, 0] ** 2 + X[:, 1] ** 2)
y = np.where((rho - R) ** 2 + X[:, 2] ** 2 < r ** 2, 1, -1)

# split 60/20/20 into training, validation, and test sets
torusTrainX, restX, torusTrainY, restY = train_test_split(X, y, test_size = 0.4, stratify = y, random_state = 2026)
torusValX, torusTestX, torusValY, torusTestY = train_test_split(restX, restY, test_size = 0.5, stratify = restY, random_state = 2026)